# Intro

In this notebook we build a country‑level travel safety classifier v3 using gradient boosting methods rather than a deep learning neural network on the same cleaned 2024 crime and safety dataset used in v2. Each country is represented by tabular features such as crime index, safety index, overall criminality scores, resilience, and LGBT legal equality indicators, and the model predicts a 5‑level ordinal risk class from very safe to very high risk. This version focuses on tree‑based ensembles that are well‑suited to small, structured datasets and provide interpretable feature importance for global travel‑safety patterns. 

# What is different in V3

Compared to v2, which used a compact fully connected neural network, v3 replaces the architecture with three gradient boosting classifiers: XGBoost, LightGBM, and scikit‑learn’s HistGradientBoostingClassifier. We keep the same cleaned dataset, features, and risk labels as v2 so that any performance differences can be attributed to the model family rather than data leakage or preprocessing changes. In addition, we introduce stratified cross‑validation and standardized evaluation metrics (accuracy, macro F1, per‑class F1) to compare the three boosting models against each other and against the v2 baseline.

# Hopes difference will change

We expect gradient boosting to achieve equal or better accuracy than the v2 neural network on this relatively small tabular dataset, while offering clearer insight into which features drive risk predictions. Tree‑based models also tend to be less sensitive to feature scaling and hyperparameter choices, which should improve robustness and make the pipeline easier to tune and extend in future versions. Finally, feature importance from these models will inform which additional data sources (e.g., temporal trends, quality‑of‑life indices) are most promising to integrate in later iterations of the country safety classifier.

In [2]:
# ============================================
# Country-Level Travel Safety Classifier v3
# Gradient Boosting Models (XGBoost, LightGBM, HistGB)
# ============================================

# Environment Setup and Imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# Gradient boosting models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


# Data Loading (2024 crime/safety dataset + lgbt)


global_path = "data/country_safety/country_safety_2024_plus_lgbt_clean.csv"
global_df = pd.read_csv(global_path)

print("Raw columns:", global_df.columns.tolist())
print(global_df.head())

Raw columns: ['country', 'crime_index', 'overall_criminality_score', 'criminal_markets_score', 'criminal_actors_score', 'resilience_score', 'safety_index', 'country_norm', 'lgbt_legal_index']
         country  crime_index  overall_criminality_score  \
0          India         44.4                       5.75   
1          China         60.8                       6.37   
2  United States         49.2                       5.67   
3      Indonesia         45.9                       6.85   
4       Pakistan         42.8                       6.03   

   criminal_markets_score  criminal_actors_score  resilience_score  \
0                    6.70                    4.8              5.42   
1                    6.53                    6.2              5.67   
2                    5.83                    5.5              7.13   
3                    6.60                    7.1              4.25   
4                    6.27                    5.8              3.96   

   safety_index   country_

In [4]:
# 5. Data Cleaning

core_cols = [
    "crime_index",
    "safety_index",
    "overall_criminality_score",
    "criminal_markets_score",
    "criminal_actors_score",
    "resilience_score",
]

# Drop rows missing core safety indices
global_df = global_df.dropna(subset=core_cols).copy()

numeric_cols = core_cols + ["lgbt_legal_index"]
for col in numeric_cols:
    if global_df[col].isna().any():
        median_val = global_df[col].median()
        global_df[col] = global_df[col].fillna(median_val)

print(global_df[numeric_cols].describe())

       crime_index  safety_index  overall_criminality_score  \
count   140.000000    140.000000                 140.000000   
mean     45.762857     54.237143                   5.333357   
std      15.254943     15.254943                   1.157298   
min      14.300000     17.900000                   2.580000   
25%      32.750000     43.900000                   4.555000   
50%      46.500000     53.500000                   5.200000   
75%      56.100000     67.250000                   6.205000   
max      82.100000     85.700000                   8.150000   

       criminal_markets_score  criminal_actors_score  resilience_score  \
count              140.000000             140.000000        140.000000   
mean                 5.203429               5.472143          5.020143   
std                  1.129403               1.322907          1.626000   
min                  1.670000               2.400000          1.500000   
25%                  4.492500               4.600000          

In [6]:
# Risk score construction and labels


# Standardize crime index
global_df["crime_z"] = (
    global_df["crime_index"] - global_df["crime_index"].mean()
) / global_df["crime_index"].std()

# Invert and standardize safety index (lower safety -> higher risk)
safety_inv = global_df["safety_index"].max() - global_df["safety_index"]
global_df["safety_z_inv"] = (safety_inv - safety_inv.mean()) / safety_inv.std()

# Combined risk score (tune weights if desired)
global_df["riskscore"] = 0.6 * global_df["crime_z"] + 0.4 * global_df["safety_z_inv"]

# Convert riskscore into 5 classes via quantiles
quantiles = global_df["riskscore"].quantile([0.2, 0.4, 0.6, 0.8]).values

def to_risk_class(x, qs=quantiles):
    if x <= qs[0]:
        return 0  # very safe
    elif x <= qs[1]:
        return 1  # safe
    elif x <= qs[2]:
        return 2  # moderate
    elif x <= qs[3]:
        return 3  # high
    else:
        return 4  # very high

global_df["riskclass"] = global_df["riskscore"].apply(to_risk_class)
print(global_df["riskclass"].value_counts().sort_index())

# Optional LGBT-based safety band (same as v2)
def queer_safety_class(x):
    if x >= 75:
        return 2
    elif x >= 40:
        return 1
    else:
        return 0

global_df["queersafetyclass"] = global_df["lgbt_legal_index"].apply(queer_safety_class)
print(global_df["queersafetyclass"].value_counts())


0    29
1    27
2    28
3    28
4    28
Name: riskclass, dtype: int64
0    52
1    51
2    37
Name: queersafetyclass, dtype: int64


In [7]:
# Feature selection and train/test split

feature_cols = [
    "crime_index",
    "safety_index",
    "overall_criminality_score",
    "criminal_markets_score",
    "criminal_actors_score",
    "resilience_score",
    "lgbt_legal_index",
    "queersafetyclass",
]

X = global_df[feature_cols].values
y = global_df["riskclass"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train_scaled.shape, "Test shape:", X_test_scaled.shape)

Train shape: (112, 8) Test shape: (28, 8)


In [8]:
# Helper: evaluation function

def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    """Fit, evaluate on test, and print metrics."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"\n=== {name} ===")
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, digits=3))

# Optional: cross-validation helper
def cross_validate_model(name, model, X, y, n_splits=5):
    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=["accuracy", "f1_macro"],
        n_jobs=-1,
        return_train_score=False,
    )
    print(f"\n{name} CV ({n_splits}-fold) - "
          f"Accuracy: {scores['test_accuracy'].mean():.3f} ± {scores['test_accuracy'].std():.3f}, "
          f"Macro F1: {scores['test_f1_macro'].mean():.3f} ± {scores['test_f1_macro'].std():.3f}")

In [9]:
# Model 1: XGBoostClassifier

xgb_clf = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=RANDOM_STATE,
)

cross_validate_model("XGBoost", xgb_clf, X_train_scaled, y_train)
evaluate_model("XGBoost", xgb_clf, X_train_scaled, y_train, X_test_scaled, y_test)

# Feature importance (gain-based)
importances_xgb = xgb_clf.feature_importances_
for col, imp in sorted(zip(feature_cols, importances_xgb), key=lambda x: -x[1]):
    print(f"XGBoost importance - {col}: {imp:.3f}")


XGBoost CV (5-fold) - Accuracy: 0.965 ± 0.051, Macro F1: 0.964 ± 0.051

=== XGBoost ===
Confusion matrix:
[[6 0 0 0 0]
 [0 5 0 0 0]
 [0 0 5 0 0]
 [0 0 0 6 0]
 [0 0 0 0 6]]

Classification report:
              precision    recall  f1-score   support

           0      1.000     1.000     1.000         6
           1      1.000     1.000     1.000         5
           2      1.000     1.000     1.000         5
           3      1.000     1.000     1.000         6
           4      1.000     1.000     1.000         6

    accuracy                          1.000        28
   macro avg      1.000     1.000     1.000        28
weighted avg      1.000     1.000     1.000        28

XGBoost importance - safety_index: 0.508
XGBoost importance - crime_index: 0.484
XGBoost importance - criminal_markets_score: 0.003
XGBoost importance - resilience_score: 0.002
XGBoost importance - lgbt_legal_index: 0.001
XGBoost importance - criminal_actors_score: 0.001
XGBoost importance - overall_criminality_s

In [10]:
# Model 2: LightGBMClassifier

lgbm_clf = LGBMClassifier(
    n_estimators=400,
    num_leaves=31,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=RANDOM_STATE,
)

cross_validate_model("LightGBM", lgbm_clf, X_train_scaled, y_train)
evaluate_model("LightGBM", lgbm_clf, X_train_scaled, y_train, X_test_scaled, y_test)

importances_lgbm = lgbm_clf.feature_importances_
for col, imp in sorted(zip(feature_cols, importances_lgbm), key=lambda x: -x[1]):
    print(f"LightGBM importance - {col}: {imp:.3f}")


LightGBM CV (5-fold) - Accuracy: 0.964 ± 0.044, Macro F1: 0.961 ± 0.049
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000063 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 253
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 8
[LightGBM] [Info] Start training from score -1.583005
[LightGBM] [Info] Start training from score -1.627456
[LightGBM] [Info] Start training from score -1.583005
[LightGBM] [Info] Start training from score -1.627456
[LightGBM] [Info] Start training from score -1.627456
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

C:\Users\ianmc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [11]:
# Model 3: HistGradientBoostingClassifier

hgb_clf = HistGradientBoostingClassifier(
    max_depth=4,
    learning_rate=0.05,
    max_iter=300,
    random_state=RANDOM_STATE,
)

# HistGB works fine on unscaled features, but we can use scaled for consistency
cross_validate_model("HistGradientBoosting", hgb_clf, X_train_scaled, y_train)
evaluate_model("HistGradientBoosting", hgb_clf, X_train_scaled, y_train, X_test_scaled, y_test)

# Feature importance via permutation (since HistGB has no native importance attribute)
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    hgb_clf,
    X_test_scaled,
    y_test,
    n_repeats=20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importances_hgb_mean = perm_result.importances_mean
for col, imp in sorted(zip(feature_cols, importances_hgb_mean), key=lambda x: -x[1]):
    print(f"HistGB perm importance - {col}: {imp:.4f}")


HistGradientBoosting CV (5-fold) - Accuracy: 0.893 ± 0.044, Macro F1: 0.891 ± 0.048

=== HistGradientBoosting ===
Confusion matrix:
[[6 0 0 0 0]
 [0 5 0 0 0]
 [0 0 5 0 0]
 [0 0 0 6 0]
 [0 0 0 0 6]]

Classification report:
              precision    recall  f1-score   support

           0      1.000     1.000     1.000         6
           1      1.000     1.000     1.000         5
           2      1.000     1.000     1.000         5
           3      1.000     1.000     1.000         6
           4      1.000     1.000     1.000         6

    accuracy                          1.000        28
   macro avg      1.000     1.000     1.000        28
weighted avg      1.000     1.000     1.000        28

HistGB perm importance - crime_index: 0.7911
HistGB perm importance - safety_index: 0.0000
HistGB perm importance - overall_criminality_score: 0.0000
HistGB perm importance - criminal_markets_score: 0.0000
HistGB perm importance - criminal_actors_score: 0.0000
HistGB perm importance - re

In [12]:
# Simple inference helper (country row -> prediction)

risk_labels = {
    0: "Very safe",
    1: "Safe",
    2: "Moderate risk",
    3: "High risk",
    4: "Very high risk",
}

# Pick best model based evaluation
best_model = xgb_clf

def predict_country_risk_v3(row):
    """
    row: pandas Series from global_df for a single country.
    returns (pred_class, label, proba_vector)
    """
    arr = row[feature_cols].values.astype(float).reshape(1, -1)
    arr_scaled = scaler.transform(arr)
    probs = best_model.predict_proba(arr_scaled)[0]
    pred_class = int(np.argmax(probs))
    return pred_class, risk_labels[pred_class], probs

sample = global_df.iloc[0]
pred_class, label, probs = predict_country_risk_v3(sample)
print("Country:", sample["country"])
print("Features:")
for c in feature_cols:
    print(f"  {c}: {sample[c]:.3f}")
print("Prediction:", pred_class, "-", label)
print("Probabilities:")
for i, p in enumerate(probs):
    print(f"  {i} {risk_labels[i]}: {p:.3f}")

Country: India
Features:
  crime_index: 44.400
  safety_index: 55.600
  overall_criminality_score: 5.750
  criminal_markets_score: 6.700
  criminal_actors_score: 4.800
  resilience_score: 5.420
  lgbt_legal_index: 61.160
  queersafetyclass: 1.000
Prediction: 2 - Moderate risk
Probabilities:
  0 Very safe: 0.006
  1 Safe: 0.005
  2 Moderate risk: 0.977
  3 High risk: 0.005
  4 Very high risk: 0.006


In [13]:
import ipynbname

nb_path = ipynbname.path()
nb_name = nb_path.name 
nb_name

'Safety_GB_model_v3.ipynb'

In [14]:
import sys
from pathlib import Path

# PDF 
!"{sys.executable}" -m nbconvert --to pdf "{nb_name}"

[NbConvertApp] Converting notebook Safety_GB_model_v3.ipynb to pdf
[NbConvertApp] Writing 207884 bytes to notebook.tex
[NbConvertApp] Building PDF
[NbConvertApp] Running xelatex 3 times: ['xelatex', 'notebook.tex', '-quiet']
[NbConvertApp] Running bibtex 1 time: ['bibtex', 'notebook']
[NbConvertApp] WARNING | b had problems, most likely because there were no citations
[NbConvertApp] PDF successfully created
[NbConvertApp] Writing 84679 bytes to Safety_GB_model_v3.pdf
